# Week 5 — Additional Models
**Internship:** IDX Exchange Data Science Program  
**Name:** Monika  
**Week:** 5  
**Dataset:** CRMLS Sold Properties, cleaned in Week 3

**Goal:** Try Decision Tree and Random Forest regressors, compare test R² against
the Week 4 Linear Regression baseline, and document how each model behaves
differently (overfitting, feature importance, non-linearity).

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_percentage_error

data_folder = r'C:\Users\monik\OneDrive - University of Illinois - Urbana\Desktop\IDX Exchange_DS\data\california'
model_df = pd.read_csv(data_folder + '\\cleaned_full.csv', parse_dates=['CloseDate_parsed'])

feature_cols = [
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeAcres',
    'PropertyAge', 'DaysOnMarket', 'Latitude', 'Longitude',
    'PoolPrivateYN', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'AssociationFee',
    'LivingArea_missing', 'BathroomsTotalInteger_missing',
    'YearBuilt_missing', 'LotSizeAcres_missing', 'DaysOnMarket_anomaly'
]
target_col = 'ClosePrice'

def get_train_test_split(frame, test_month, window_months):
    frame = frame.copy()
    frame['YearMonth'] = frame['CloseDate_parsed'].dt.to_period('M')
    test_df = frame[frame['YearMonth'] == test_month]
    train_start = test_month - window_months
    train_df = frame[(frame['YearMonth'] >= train_start) & (frame['YearMonth'] < test_month)]
    return train_df.drop(columns='YearMonth'), test_df.drop(columns='YearMonth')

## 1. Re-establish the Baseline Split
Re-running the Week 4 window sweep with Linear Regression so this notebook picks
the same best window on its own (self-contained), then locking that window in
for all three models so the R² comparison is apples-to-apples.

In [2]:
test_month = pd.Period('2026-06', freq='M')
window_options = [3, 6, 9, 12, 18, 24]

sweep_results = []
for window in window_options:
    train_df, test_df = get_train_test_split(model_df, test_month, window)
    if len(train_df) < 50 or len(test_df) < 10:
        continue
    X_train, y_train = train_df[feature_cols], train_df[target_col]
    X_test, y_test = test_df[feature_cols], test_df[target_col]
    scaler = StandardScaler()
    lr = LinearRegression().fit(scaler.fit_transform(X_train), y_train)
    r2 = r2_score(y_test, lr.predict(scaler.transform(X_test)))
    sweep_results.append({'window_months': window, 'R2': r2})

sweep_df = pd.DataFrame(sweep_results)
BEST_WINDOW = int(sweep_df.loc[sweep_df['R2'].idxmax(), 'window_months'])
print(f'Locked-in window for comparison: {BEST_WINDOW} months')

train_df, test_df = get_train_test_split(model_df, test_month, BEST_WINDOW)
X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

Locked-in window for comparison: 3 months


## 2. Baseline: Linear Regression
Same as Week 4, refit here just so it lives in the same comparison table below.

In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LinearRegression().fit(X_train_scaled, y_train)
lr_train_r2 = r2_score(y_train, lr_model.predict(X_train_scaled))
lr_test_r2 = r2_score(y_test, lr_model.predict(X_test_scaled))
lr_test_mape = mean_absolute_percentage_error(y_test, lr_model.predict(X_test_scaled))

print(f'Linear Regression — Train R²: {lr_train_r2:.4f} | Test R²: {lr_test_r2:.4f}')

Linear Regression — Train R²: 0.4803 | Test R²: 0.4810


## 3. Decision Tree Regressor
Trees don't need scaled features, so I'm using the unscaled X_train/X_test here.
I'm tuning `max_depth` on a **held-out slice of the training data** (not the
test set) — tuning on test would make this comparison unfair to the baseline,
since I'd effectively be peeking at the answer.

In [4]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

depth_options = [3, 5, 8, 10, 15, None]
depth_results = []
for depth in depth_options:
    dt = DecisionTreeRegressor(max_depth=depth, random_state=42).fit(X_tr, y_tr)
    val_r2 = r2_score(y_val, dt.predict(X_val))
    depth_results.append({'max_depth': depth, 'val_R2': val_r2})

depth_df = pd.DataFrame(depth_results)
print(depth_df)

best_depth = depth_df.loc[depth_df['val_R2'].idxmax(), 'max_depth']
print(f'Best max_depth: {best_depth}')

   max_depth    val_R2
0        3.0  0.416864
1        5.0  0.597126
2        8.0  0.745225
3       10.0  0.791687
4       15.0  0.785420
5        NaN  0.774802
Best max_depth: 10.0


In [10]:
best_depth = depth_df.loc[depth_df['val_R2'].idxmax(), 'max_depth']

if pd.notna(best_depth):
    best_depth = int(best_depth)

print(best_depth)

10


In [11]:
print(best_depth)
print(type(best_depth))

10
<class 'int'>


In [12]:
dt_model = DecisionTreeRegressor(max_depth=best_depth, random_state=42).fit(X_train, y_train)
dt_train_r2 = r2_score(y_train, dt_model.predict(X_train))
dt_test_r2 = r2_score(y_test, dt_model.predict(X_test))
dt_test_mape = mean_absolute_percentage_error(y_test, dt_model.predict(X_test))

print(f'Decision Tree — Train R²: {dt_train_r2:.4f} | Test R²: {dt_test_r2:.4f}')

Decision Tree — Train R²: 0.8577 | Test R²: 0.7846


## 4. Random Forest Regressor
Same idea, but averaging across many trees should reduce the overfitting a
single deep Decision Tree tends to show. Using a fixed n_estimators/max_depth
here rather than a full grid search, to keep this manageable — worth revisiting
with GridSearchCV later if this becomes the model I keep.

In [13]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
).fit(X_train, y_train)

rf_train_r2 = r2_score(y_train, rf_model.predict(X_train))
rf_test_r2 = r2_score(y_test, rf_model.predict(X_test))
rf_test_mape = mean_absolute_percentage_error(y_test, rf_model.predict(X_test))

print(f'Random Forest — Train R²: {rf_train_r2:.4f} | Test R²: {rf_test_r2:.4f}')

Random Forest — Train R²: 0.9398 | Test R²: 0.8731


## 5. Comparison Table